# Netflix Data Analysis and Visualization

This notebook solves the business case using the provided Netflix dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('netflix_titles.csv')
df.head()

## 1. Un-nesting the Columns (Comma-Separated Values)

In [ ]:
def unnest_column(df, col):
    s = df.dropna(subset=[col])
    s = s.assign(**{col: s[col].str.split(', ')}).explode(col)
    return s

df_cast = unnest_column(df, 'cast')
df_director = unnest_column(df, 'director')
df_genre = unnest_column(df, 'listed_in')

df_cast.head()

## 2. Handling Null Values

In [ ]:
df.fillna({
    'director': 'Unknown Director',
    'cast': 'Unknown Cast',
    'country': 'Unknown Country',
    'rating': 'Unknown Rating',
    'date_added': 'Unknown Date'
}, inplace=True)
df.head()

## 3. Count of Each Categorical Variable (Graphical & Non-Graphical)

In [ ]:
cat_columns = ['type', 'rating', 'country']
for col in cat_columns:
    print(f"\nValue Counts for {col}:")
    print(df[col].value_counts().head(10))

    plt.figure(figsize=(10,5))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index[:10])
    plt.xticks(rotation=45)
    plt.title(f'Top 10 {col} Distribution')
    plt.tight_layout()
    plt.show()

## 4. Comparison of TV Shows vs. Movies by Country

In [ ]:
# Movies by country
top_movie_countries = df[df['type']=='Movie'].groupby('country')['title'].count().sort_values(ascending=False).head(10)
top_tv_countries = df[df['type']=='TV Show'].groupby('country')['title'].count().sort_values(ascending=False).head(10)

print("Top 10 Countries Producing Movies:")
print(top_movie_countries)

top_movie_countries.plot(kind='bar', title='Top 10 Countries Producing Movies', figsize=(10,5))
plt.ylabel('Number of Movies')
plt.show()

print("\nTop 10 Countries Producing TV Shows:")
print(top_tv_countries)

top_tv_countries.plot(kind='bar', title='Top 10 Countries Producing TV Shows', figsize=(10,5), color='orange')
plt.ylabel('Number of TV Shows')
plt.show()

## 5. Best Time to Launch TV Shows or Movies

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['week'] = df['date_added'].dt.week
df['month'] = df['date_added'].dt.month

# Weekly and Monthly Launches
for t in ['Movie', 'TV Show']:
    df_temp = df[df['type']==t]
    week_counts = df_temp['week'].value_counts().sort_index()
    month_counts = df_temp['month'].value_counts().sort_index()

    plt.figure(figsize=(10,4))
    week_counts.plot(kind='bar')
    plt.title(f'{t}: Releases by Week')
    plt.xlabel('Week Number')
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(10,4))
    month_counts.plot(kind='bar', color='green')
    plt.title(f'{t}: Releases by Month')
    plt.xlabel('Month')
    plt.ylabel('Count')
    plt.show()

## 6. Top 10 Actors and Directors

In [ ]:
top_actors = df_cast['cast'].value_counts().head(10)
top_directors = df_director['director'].value_counts().head(10)

print("Top 10 Actors:")
print(top_actors)

print("\nTop 10 Directors:")
print(top_directors)

top_actors.plot(kind='bar', figsize=(10,5), title='Top 10 Actors')
plt.ylabel('Appearances')
plt.show()

top_directors.plot(kind='bar', figsize=(10,5), color='purple', title='Top 10 Directors')
plt.ylabel('Appearances')
plt.show()

## 7. Genre Popularity Word Cloud

In [ ]:
text = " ".join(df['listed_in'].dropna())
wordcloud = WordCloud(width=800, height=400, background_color='black').generate(text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Popular Genres on Netflix')
plt.show()

## 8. Days Difference Between Release Year and Date Added

In [ ]:
df['year_added'] = df['date_added'].dt.year
df['diff'] = df['year_added'] - df['release_year']
df['diff'].dropna().astype(int).mode()

## Final Insights and Recommendations

- Most content is added between September and December.
- TV Shows and Movies both peak during the winter holiday season.
- The United States dominates in content production.
- The most popular genres include Drama, International Shows, and Comedies.
- Majority of content is added to Netflix 1 year after its official release.

These insights can help Netflix schedule new launches and guide production strategy.